In [1]:
# Data analysis
import pandas as pd
import numpy as np

# Plotting and Correlation Maths
import seaborn as sns
import scipy as sci

# Simple model development
import sklearn as sk
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import plotly as px

import os


In [2]:
file_path = '/Users/hunterberberich/Downloads/cleaned_data (1).csv'
file_path2 = '/Users/hunterberberich/Desktop/Data/merged_df.csv'
cleaned_df = pd.read_csv(file_path)
merged_df = pd.read_csv(file_path2)

In [4]:
#Unqiue Key for merge
cleaned_df['unique_key'] = cleaned_df['tempmax'].astype(str) + '_' + cleaned_df['tempmin'].astype(str)
merged_df['unique_key'] = merged_df['tempmax'].astype(str) + '_' + merged_df['tempmin'].astype(str)
merged_df_unique = merged_df.drop_duplicates(subset='unique_key', keep='first')
survive_dict = dict(zip(merged_df_unique['unique_key'], merged_df_unique['survive']))

#'survive' column to cleaned_df by matching 'unique_key'
cleaned_df['survive'] = cleaned_df['unique_key'].map(survive_dict)

survive_df = cleaned_df.copy()

print("Survive DataFrame Head:\n", survive_df.head())

# Check data for accuracy
print(f"Original cleaned_df shape: {cleaned_df.shape}")
print(f"New survive_df shape: {survive_df.shape}")


missing_values = survive_df['survive'].isnull().sum()
print(f"Number of missing values in 'survive' column: {missing_values}")

spot_check_indices = [0, 10, 50]  # Example indices to check
print("Spot check specific rows in survive_df:")
print(survive_df.loc[spot_check_indices, ['tempmax', 'tempmin', 'survive', 'unique_key']])
print("Corresponding rows in merged_df:")
print(merged_df.loc[merged_df['unique_key'].isin(survive_df.loc[spot_check_indices, 'unique_key']), ['tempmax', 'tempmin', 'survive', 'unique_key']])


missing_keys = survive_df[~survive_df['unique_key'].isin(merged_df_unique['unique_key'])]
print(f"Number of keys in survive_df not found in merged_df: {missing_keys.shape[0]}")


survive_df = survive_df.drop(columns=['unique_key'])

result_file_path = '/Users/hunterberberich/Downloads/cleaned_data_with_survive2.csv'
survive_df.to_csv(result_file_path, index=False)

Survive DataFrame Head:
    tempmax  tempmin  temp  feelslikemax  feelslikemin  feelslike   dew  \
0     26.2     -4.2  10.1          18.1         -16.6        0.6  -3.8   
1     35.8     28.0  32.5          34.7          24.1       26.8  26.6   
2     15.3      2.5   9.2          14.6          -6.0        3.5   0.5   
3     15.3      2.5   9.2          14.6          -6.0        3.5   0.5   
4     15.3      2.5   9.2          14.6          -6.0        3.5   0.5   

   humidity  precip  precipcover  ...  preciptype_freezingrain,snow,ice  \
0      57.2     0.0          0.0  ...                                 0   
1      78.9     0.0          0.0  ...                                 0   
2      68.0     0.0          0.0  ...                                 0   
3      68.0     0.0          0.0  ...                                 0   
4      68.0     0.0          0.0  ...                                 0   

   preciptype_none  preciptype_rain  preciptype_rain,freezingrain  \
0         

In [5]:
# Remove 'U' to make 'survive' Binary 
survive_df['survive'] = survive_df['survive'].replace('U', pd.NA)
survive_df['survive'] = survive_df['survive'].map({'Y': 1, 'N': 0})

survive_df

,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,precip,precipcover,...,"preciptype_freezingrain,snow","preciptype_freezingrain,snow,ice",preciptype_none,preciptype_rain,"preciptype_rain,freezingrain","preciptype_rain,freezingrain,snow","preciptype_rain,snow","preciptype_rain,snow,ice",preciptype_snow,survive
0,26.2,-4.2,10.1,18.1,-16.6,0.6,-3.8,57.2,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
1,35.8,28.0,32.5,34.7,24.1,26.8,26.6,78.9,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
2,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
3,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
4,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3679,68.8,58.9,61.6,68.8,58.9,61.6,56.7,84.8,1.526,83.33,...,0,0,0,1,0,0,0,0,0,NaN
3680,59.4,46.8,50.8,59.4,39.8,46.5,46.3,84.5,0.266,83.33,...,0,0,0,1,0,0,0,0,0,1.0
3681,59.2,44.5,51.1,59.2,37.4,47.9,33.0,51.5,0.019,12.50,...,0,0,0,1,0,0,0,0,0,1.0
3682,81.1,45.0,64.1,79.9,39.9,62.9,39.5,42.6,0.001,4.17,...,0,0,0,1,0,0,0,0,0,1.0


In [6]:
survive_df = survive_df.dropna(subset=['survive'])

survive_df

,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,precip,precipcover,...,"preciptype_freezingrain,snow","preciptype_freezingrain,snow,ice",preciptype_none,preciptype_rain,"preciptype_rain,freezingrain","preciptype_rain,freezingrain,snow","preciptype_rain,snow","preciptype_rain,snow,ice",preciptype_snow,survive
0,26.2,-4.2,10.1,18.1,-16.6,0.6,-3.8,57.2,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
1,35.8,28.0,32.5,34.7,24.1,26.8,26.6,78.9,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
2,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
3,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
4,15.3,2.5,9.2,14.6,-6.0,3.5,0.5,68.0,0.000,0.00,...,0,0,1,0,0,0,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3678,65.1,57.1,60.6,65.1,57.1,60.6,54.5,80.9,0.179,62.50,...,0,0,0,1,0,0,0,0,0,1.0
3680,59.4,46.8,50.8,59.4,39.8,46.5,46.3,84.5,0.266,83.33,...,0,0,0,1,0,0,0,0,0,1.0
3681,59.2,44.5,51.1,59.2,37.4,47.9,33.0,51.5,0.019,12.50,...,0,0,0,1,0,0,0,0,0,1.0
3682,81.1,45.0,64.1,79.9,39.9,62.9,39.5,42.6,0.001,4.17,...,0,0,0,1,0,0,0,0,0,1.0


In [7]:
result_file_path = '/Users/hunterberberich/Downloads/survive_cleaned.csv'
survive_df.to_csv(result_file_path, index=False)
